In [1]:
import rawpy
import cv2
import numpy as np
from PIL import Image
from ultralytics import YOLO
import sys
from dotenv import load_dotenv
import os
from pathlib import Path
load_dotenv()


MODELE_IA = 'yolov8s-seg.pt'

CLASS_ID_OISEAU = 14 

def charger_et_developper_raf_16bit(chemin_raf):
    """
    Ouvre le fichier .RAF et le convertit en une image RGB 16 bits (0-65535).
    Le 16 bits retient beaucoup plus d'informations de couleur que le 8 bits standard.
    """
    print(f"-> 1. Lecture et développement du fichier RAW (16-bit) : {chemin_raf}")
    try:
        with rawpy.imread(chemin_raf) as raw:
           
            rgb_16bit = raw.postprocess(
                use_camera_wb=True, 
                bright=1.0,          
                user_sat=None,       
                no_auto_bright=False,
                output_bps=16        
            )
        return rgb_16bit
    except Exception as e:
        print(f"Erreur lors du chargement du RAW : {e}")
        sys.exit(1)

def generer_masque_oiseau_ia(image_rgb_16bit):
    """
    Utilise YOLOv8 pour trouver les oiseaux et générer un masque de segmentation.
    """
    print(f"-> 2. Analyse IA (YOLOv8) pour trouver l'oiseau")
    
    img_8bit_pour_ia = (image_rgb_16bit / 256).astype('uint8')
    
    model = YOLO(MODELE_IA)
    
    results = model.predict(img_8bit_pour_ia, verbose=False, conf=0.4)
    result = results[0]

    h, w = image_rgb_16bit.shape[:2]
    masque_final = np.zeros((h, w), dtype=np.uint8)
    oiseau_trouve = False

    if result.masks is not None:
        for i, box in enumerate(result.boxes):
            class_id = int(box.cls[0])
            if class_id == CLASS_ID_OISEAU:
                oiseau_trouve = True
                mask_data = result.masks.data[i].cpu().numpy()
                mask_resized = cv2.resize(mask_data, (w, h), interpolation=cv2.INTER_LINEAR)
                masque_final = np.maximum(masque_final, mask_resized)

    if not oiseau_trouve:
        print("ATTENTION : Aucun oiseau détecté par l'IA. L'amélioration ciblée sera ignorée.")
        return masque_final

    print("   Oiseau(x) identifié(s) et masqué(s) avec succès.")
    return (masque_final * 255).astype(np.uint8)

def traiter_image_haute_qualite(image_rgb_16bit, mask_8bit):
   
    print("-> 3. Application des traitements (Correction 16-bit -> Float32)...")
    
    img_bgr_16 = cv2.cvtColor(image_rgb_16bit, cv2.COLOR_RGB2BGR)

    blurred_16 = cv2.GaussianBlur(img_bgr_16, (0, 0), sigmaX=3)
    sharpened_16 = cv2.addWeighted(img_bgr_16, 1.8, blurred_16, -0.8, 0)

    mask_3c_norm = cv2.cvtColor(mask_8bit, cv2.COLOR_GRAY2BGR).astype(np.float32) / 255.0
    img_bgr_16_float = img_bgr_16.astype(np.float32)
    sharpened_16_float = sharpened_16.astype(np.float32)
    
    merged_float = (sharpened_16_float * mask_3c_norm) + (img_bgr_16_float * (1.0 - mask_3c_norm))

    img_normalized = merged_float / 65535.0
    
    lab_f32 = cv2.cvtColor(img_normalized, cv2.COLOR_BGR2Lab)
    l_channel, a_channel, b_channel = cv2.split(lab_f32)

    l_u16 = (l_channel * (65535.0 / 100.0)).astype(np.uint16)

    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    l_enhanced_u16 = clahe.apply(l_u16)

    l_enhanced_f32 = l_enhanced_u16.astype(np.float32) * (100.0 / 65535.0)

    lab_final_f32 = cv2.merge((l_enhanced_f32, a_channel, b_channel))
    final_bgr_f32 = cv2.cvtColor(lab_final_f32, cv2.COLOR_Lab2BGR)

    final_bgr_16 = np.clip(final_bgr_f32 * 65535.0, 0, 65535).astype(np.uint16)

    return final_bgr_16

def sauvegarder_tiff_hq(image_bgr_16bit, chemin_sortie):
    """
    Sauvegarde en TIFF 16-bit haute qualité en utilisant OpenCV.
    OpenCV gère mieux les tableaux NumPy 16-bit multi-canaux pour le format TIFF.
    """
    print(f"-> 4. Sauvegarde du TIFF 16-bit (Haute Qualité) : {chemin_sortie}")
    
    try:

        params = [cv2.IMWRITE_TIFF_COMPRESSION, 32946] 
        
        succes = cv2.imwrite(chemin_sortie, image_bgr_16bit, params)
        
        if succes:
            print(f"Fichier enregistré avec succès en 16-bit !")
        else:
            print("Échec de l'enregistrement du fichier.")
            
    except Exception as e:
        print(f"Erreur lors de la sauvegarde : {e}")


In [2]:
if __name__ == "__main__":
    base_path = os.getenv("BASE_PATH")
    fichier_entree_raf = f"{base_path}data/inputs/DSCF4495.RAF"
    fichier_sortie_tiff = f"{base_path}data/outputs/DSCF4495_postttt.tiff" 

    # 1. Charger le RAW
    rgb_16 = charger_et_developper_raf_16bit(fichier_entree_raf)
    
    # 2. IA : Masque de l'oiseau
    masque_ia = generer_masque_oiseau_ia(rgb_16)
    
    # 3. Traitement (Netteté ciblée + Lab CLAHE corrigé)
    # Assurez-vous d'utiliser la version "float32" de cette fonction donnée précédemment
    resultat_bgr_16 = traiter_image_haute_qualite(rgb_16, masque_ia)
    
    # 4. Sauvegarde directe via OpenCV
    sauvegarder_tiff_hq(resultat_bgr_16, fichier_sortie_tiff)

-> 1. Lecture et développement du fichier RAW (16-bit) : C:/Users/savma/Projects/ttt_image/data/inputs/DSCF4495.RAF
-> 2. Analyse IA (YOLOv8) pour trouver l'oiseau
   Oiseau(x) identifié(s) et masqué(s) avec succès.
-> 3. Application des traitements (Correction 16-bit -> Float32)...
-> 4. Sauvegarde du TIFF 16-bit (Haute Qualité) : C:/Users/savma/Projects/ttt_image/data/outputs/DSCF4495_postttt.tiff
Fichier enregistré avec succès en 16-bit !
